In [62]:
import pandas as pd
import numpy as np
import glob


In [63]:
# Load all coins
dfs = []
for file in glob.glob("coin_*.csv"):
    df = pd.read_csv(file)
    df["coin"] = file.replace("coin_", "").replace(".csv", "")
    dfs.append(df)

In [64]:
data = pd.concat(dfs).reset_index(drop=True)
data["Date"] = pd.to_datetime(data["Date"])


In [65]:
# Features per coin
results = []
for coin, group in data.groupby("coin"):
    g = group.sort_values("Date").copy()
    g["ret_1d"]     = g["Close"].pct_change(1)
    g["ret_7d"]     = g["Close"].pct_change(7)
    g["ma_7"]       = g["Close"].rolling(7).mean()
    g["ma_30"]      = g["Close"].rolling(30).mean()
    g["ma_ratio"]   = g["ma_7"] / g["ma_30"]
    g["volatility"] = g["ret_1d"].rolling(7).std()
    g["target"]     = g["Close"].pct_change(1).shift(-1)
    results.append(g)

data = pd.concat(results)

In [66]:
# Numerical coin ID
data["coin_id"] = pd.Categorical(data["coin"]).codes.astype(int)


# Drop NaNs, sort by date then coin
data = (data
        
        .dropna()
        .sort_values(["Date", "coin_id"])
        .reset_index(drop=True))

print(data.shape)


(36392, 15)


In [67]:
data["direction"] = (data["target"] > 0).astype(int)

In [68]:
data["coin_id"].unique()

array([ 2, 11, 22,  7, 12, 16, 17, 13,  9, 10,  8,  1, 18,  4,  3, 19,  6,
       21,  5, 15, 14, 20,  0])

In [71]:
data["day_of_week"] = data["Date"].dt.dayofweek   # 0=Monday, 6=Sunday
data["month"]       = data["Date"].dt.month

In [72]:
data.drop(columns=["target","Date"],inplace=True)

In [69]:
data.value_counts("coin")

coin
Bitcoin           2961
Litecoin          2961
XRP               2863
Dogecoin          2730
Monero            2572
Stellar           2497
Tether            2288
NEM               2258
Ethereum          2130
Iota              1454
EOS               1436
BinanceCoin       1412
Tron              1362
ChainLink         1355
Cardano           1344
USDCoin            972
CryptocomCoin      905
WrappedBitcoin     858
Cosmos             815
Solana             422
Polkadot           290
Uniswap            262
Aave               245
Name: count, dtype: int64

In [73]:
data.head(10)

,Open,High,Low,Close,Volume,coin,ret_1d,ret_7d,ma_7,ma_30,ma_ratio,volatility,coin_id,direction,day_of_week,month
0,129.770004,130.580002,125.599998,129.000000,0.0,Bitcoin,-0.005742,0.049805,129.713426,120.440532,1.076991,0.025417,2,1,1,5
1,3.102800,3.130000,3.019420,3.092130,0.0,Litecoin,-0.002831,0.001743,3.148901,3.320466,0.948331,0.027956,11,1,1,5
2,129.000000,132.589996,127.662003,132.300003,0.0,Bitcoin,0.025581,0.067891,130.914998,120.032533,1.090663,0.026358,2,0,2,5
3,3.092130,3.123270,3.019660,3.095820,0.0,Litecoin,0.001193,-0.007992,3.145339,3.277530,0.959667,0.027589,11,0,2,5
4,132.250000,132.250000,127.000000,128.798996,0.0,Bitcoin,-0.026463,0.016567,131.214855,119.692499,1.096266,0.028765,2,1,3,5
5,3.077460,3.092780,2.884170,2.971390,0.0,Litecoin,-0.040193,-0.066024,3.115331,3.233360,0.963497,0.029435,11,0,3,5
6,128.798996,129.899994,126.400002,129.000000,0.0,Bitcoin,0.001561,-0.031532,130.614855,120.092833,1.087616,0.019366,2,1,4,5
7,2.971390,2.988230,2.688820,2.956680,0.0,Litecoin,-0.004951,-0.070074,3.083503,3.205215,0.962027,0.029266,11,0,4,5
8,128.815002,129.779999,127.198997,129.300003,0.0,Bitcoin,0.002326,-0.020306,130.231999,120.895833,1.077225,0.019383,2,0,5,6
9,2.942140,2.974560,2.802700,2.826500,0.0,Litecoin,-0.044029,-0.093690,3.041761,3.187033,0.954418,0.031957,11,0,5,6


In [57]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 36392 entries, 0 to 36391
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        36392 non-null  datetime64[us]
 1   Open        36392 non-null  float64       
 2   High        36392 non-null  float64       
 3   Low         36392 non-null  float64       
 4   Close       36392 non-null  float64       
 5   Volume      36392 non-null  float64       
 6   coin        36392 non-null  str           
 7   ret_1d      36392 non-null  float64       
 8   ret_7d      36392 non-null  float64       
 9   ma_7        36392 non-null  float64       
 10  ma_30       36392 non-null  float64       
 11  ma_ratio    36392 non-null  float64       
 12  volatility  36392 non-null  float64       
 13  coin_id     36392 non-null  int64         
 14  direction   36392 non-null  int64         
dtypes: datetime64[us](1), float64(11), int64(2), str(1)
memory usage: 4.2 MB


In [76]:
data.describe()

,Open,High,Low,Close,Volume,ret_1d,ret_7d,ma_7,ma_30,ma_ratio,volatility,coin_id,direction,day_of_week,month
count,36392.000000,36392.000000,36392.000000,36392.000000,3.639200e+04,36392.000000,36392.000000,36392.000000,36392.000000,36392.000000,36392.000000,36392.000000,36392.000000,36392.000000,36392.000000
mean,998.969007,1030.131994,966.092395,1000.759037,3.075231e+09,0.004392,0.034712,995.160720,973.378400,1.028887,0.051340,11.042372,0.492883,3.000824,6.432897
std,5128.092232,5290.704683,4946.083198,5133.580972,1.201145e+10,0.072576,0.261011,5109.891902,5016.312861,0.194089,0.050011,6.268003,0.499956,2.000797,3.458012
min,0.000086,0.000089,0.000079,0.000086,0.000000e+00,-0.460055,-0.582141,0.000090,0.000095,0.519222,0.000000,0.000000,0.000000,0.000000,1.000000
25%,0.073219,0.076545,0.070314,0.073340,5.786196e+06,-0.021660,-0.062086,0.073221,0.072081,0.934777,0.023724,7.000000,0.000000,1.000000,3.000000
50%,1.001190,1.008833,0.999839,1.001163,9.084551e+07,0.000000,0.000000,1.001174,1.001219,1.000000,0.041190,11.000000,0.000000,3.000000,6.000000
75%,31.483850,33.027184,29.942456,31.565076,9.877728e+08,0.023640,0.077551,31.549912,30.334852,1.082332,0.065131,16.000000,1.000000,5.000000,9.000000
max,63523.754869,64863.098908,62208.964366,63503.457930,3.509679e+11,3.555712,8.088124,61754.598907,58245.488510,3.265739,1.372726,22.000000,1.000000,6.000000,12.000000


In [81]:
data.drop(columns=["coin"]).corr()

,Open,High,Low,Close,Volume,ret_1d,ret_7d,ma_7,ma_30,ma_ratio,volatility,coin_id,direction,day_of_week,month
Open,1.000000,0.999105,0.999180,0.998969,0.285778,-0.006848,-0.006606,0.998850,0.990787,0.009076,-0.042919,-0.029506,0.011706,0.000275,-0.049919
High,0.999105,1.000000,0.998615,0.999075,0.286172,-0.003989,-0.005828,0.998288,0.990280,0.009320,-0.041939,-0.028923,0.011244,-0.000264,-0.050322
Low,0.999180,0.998615,1.000000,0.999476,0.283880,-0.003504,-0.005441,0.997936,0.989631,0.009056,-0.044311,-0.029659,0.011250,0.000674,-0.049123
Close,0.998969,0.999075,0.999476,1.000000,0.285449,-0.001295,-0.004902,0.998109,0.989784,0.009478,-0.042940,-0.029488,0.011082,0.000067,-0.049856
Volume,0.285778,0.286172,0.283880,0.285449,1.000000,0.003221,0.006016,0.286181,0.280954,0.018137,-0.086682,-0.013718,0.007229,-0.010093,-0.074692
ret_1d,-0.006848,-0.003989,-0.003504,-0.001295,0.003221,1.000000,0.406406,-0.005971,-0.006836,0.115311,0.187854,-0.005263,-0.055265,-0.006446,-0.010249
ret_7d,-0.006606,-0.005828,-0.005441,-0.004902,0.006016,0.406406,1.000000,-0.009641,-0.014874,0.574995,0.487430,-0.006474,-0.002672,0.000507,-0.017967
ma_7,0.998850,0.998288,0.997936,0.998109,0.286181,-0.005971,-0.009641,1.000000,0.993537,0.007219,-0.042613,-0.029584,0.011191,0.000133,-0.050068
ma_30,0.990787,0.990280,0.989631,0.989784,0.280954,-0.006836,-0.014874,0.993537,1.000000,-0.006507,-0.041615,-0.029993,0.010951,0.000149,-0.048984
ma_ratio,0.009076,0.009320,0.009056,0.009478,0.018137,0.115311,0.574995,0.007219,-0.006507,1.000000,0.394979,-0.022133,0.018635,0.000241,-0.055238


In [83]:
data.to_csv("output-1.0.csv")